# Wellness Tourism MLOps Project

## Data Registration
- Create folder structure
- Register dataset to Hugging Face

In [29]:
import os
os.makedirs("mlops_project", exist_ok=True)
os.makedirs("mlops_project/data", exist_ok=True)
os.makedirs("mlops_project/model_building", exist_ok=True)

Please manually place `tourism.csv` into the `mlops_project/data` folder if not already there.

In [30]:
%%writefile mlops_project/model_building/data_register.py
from huggingface_hub import HfApi, create_repo
import os

repo_id = "ashwindatasense/wellness-tourism-dataset"
repo_type = "dataset"

api = HfApi(token=os.getenv("HF_TOKEN"))

try:
    api.repo_info(repo_id=repo_id, repo_type=repo_type)
    print(f"Space '{repo_id}' already exists.")
except Exception:
    print(f"Space '{repo_id}' not found. Creating...")
    create_repo(repo_id=repo_id, repo_type=repo_type, private=False)

api.upload_folder(
    folder_path="mlops_project/data",
    repo_id=repo_id,
    repo_type=repo_type
)
print("Data uploaded successfully!")

Overwriting mlops_project/model_building/data_register.py


## Data Preparation
Script for CI/CD pipeline to load, clean, split, and upload train/test datasets.

In [31]:
%%writefile mlops_project/model_building/prep.py
import pandas as pd
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi
import os

api = HfApi(token=os.getenv("HF_TOKEN"))
repo_id = "ashwindatasense/wellness-tourism-dataset"

# Construct direct download URL for the raw dataset
base_url = f"https://huggingface.co/datasets/{repo_id}/resolve/main"

# Load dataset using HTTPS instead of hf://
print("Loading raw dataset from Hugging Face Hub...")
df = pd.read_csv(f"{base_url}/tourism.csv")

# Clean
if 'CustomerID' in df.columns:
    df.drop(columns=['CustomerID'], inplace=True)

# Split
target_col = 'ProdTaken'
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Save
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)
X_train.to_csv(f"{data_dir}/X_train.csv", index=False)
X_test.to_csv(f"{data_dir}/X_test.csv", index=False)
y_train.to_csv(f"{data_dir}/y_train.csv", index=False)
y_test.to_csv(f"{data_dir}/y_test.csv", index=False)

# Upload
for file_name in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    api.upload_file(
        path_or_fileobj=f"{data_dir}/{file_name}",
        path_in_repo=file_name,
        repo_id=repo_id,
        repo_type="dataset"
    )
print("Data prep complete and split files uploaded.")

Overwriting mlops_project/model_building/prep.py


## Model Training
### Experimentation and Tracking (Development Environment)

In [32]:
# 1. Install pyngrok and other dependencies
!pip install pyngrok mlflow scikit-learn xgboost -q

# 2. Force the runtime to see the new packages without a full restart
import site
from importlib import reload
site.main()

try:
    import pyngrok
    print("pyngrok successfully detected.")
except ImportError:
    print("pyngrok not detected. If the next cell fails, go to Runtime -> Restart session.")

pyngrok successfully detected.


In [33]:
from pyngrok import ngrok
import subprocess
import mlflow
import os

# Set your Ngrok Token here
NGROK_TOKEN = "<YOUR_NGROK_AUTHTOKEN_HERE>"

if not NGROK_TOKEN or NGROK_TOKEN == "<YOUR_NGROK_AUTHTOKEN_HERE>":
    print("❌ Please set your ngrok authtoken.")
else:
    try:
        ngrok.set_auth_token(NGROK_TOKEN)

        print("🚀 Starting MLflow UI...")
        process = subprocess.Popen(["mlflow", "ui", "--port", "5000", "--backend-store-uri", "sqlite:///mlflow.db"])

        public_url = ngrok.connect(5000).public_url
        print(f"✅ MLflow UI is now live at: {public_url}")

        mlflow.set_tracking_uri("sqlite:///mlflow.db")
        mlflow.set_experiment("Wellness_Tourism_MLOps")
        print("Experiment set to: Wellness_Tourism_MLOps")
    except Exception as e:
        print(f"❌ Failed to connect ngrok: {e}")

🚀 Starting MLflow UI...
✅ MLflow UI is now live at: https://pang-crust-lance.ngrok-free.dev
Experiment set to: Wellness_Tourism_MLOps


### Step: Run Data Pipelines
Before training the model, we need to run the registration and preparation scripts to upload the dataset to the Hugging Face Hub.

In [34]:
import os

# Set your Hugging Face Token here
os.environ["HF_TOKEN"] = "<YOUR_HF_TOKEN_HERE>"

try:
    print("Running data_register.py...")
    !python mlops_project/model_building/data_register.py

    print("\nRunning prep.py...")
    !python mlops_project/model_building/prep.py
except Exception as e:
    print(f"❌ Error: {e}")

✅ HF_TOKEN found in secrets.
Running data_register.py...
Space 'ashwindatasense/wellness-tourism-dataset' already exists.
No files have been modified since last commit. Skipping to prevent empty commit.
Data uploaded successfully!

Running prep.py...
Loading raw dataset from Hugging Face Hub...
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.
Data prep complete and split files uploaded.


In [35]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score
import mlflow
import os

repo_id = "ashwindatasense/wellness-tourism-dataset"

# Construct direct download URLs from Hugging Face
base_url = f"https://huggingface.co/datasets/{repo_id}/resolve/main"

print("Loading train and test splits from Hugging Face...")
try:
    X_train = pd.read_csv(f"{base_url}/X_train.csv")
    X_test = pd.read_csv(f"{base_url}/X_test.csv")
    y_train = pd.read_csv(f"{base_url}/y_train.csv").squeeze()
    y_test = pd.read_csv(f"{base_url}/y_test.csv").squeeze()
    print("Data loaded successfully from Hugging Face Hub.")
except Exception as e:
    print(f"Could not load from Hub ({e}). Falling back to local files...")
    X_train = pd.read_csv("data/X_train.csv")
    X_test = pd.read_csv("data/X_test.csv")
    y_train = pd.read_csv("data/y_train.csv").squeeze()
    y_test = pd.read_csv("data/y_test.csv").squeeze()
    print("Data loaded from local directory.")

numeric_features = ['Age', 'DurationOfPitch', 'NumberOfPersonVisiting', 'NumberOfFollowups',
                    'PreferredPropertyStar', 'NumberOfTrips', 'PitchSatisfactionScore',
                    'NumberOfChildrenVisiting', 'MonthlyIncome']
categorical_features = ['TypeofContact', 'Occupation', 'Gender', 'ProductPitched', 'MaritalStatus', 'Designation']
passthrough_features = ['CityTier', 'Passport', 'OwnCar']

numeric_transformer = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))
passthrough_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'))

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features),
    (passthrough_transformer, passthrough_features)
)

xgb_model = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid = {
    'xgbclassifier__n_estimators': [50, 100],
    'xgbclassifier__max_depth': [3, 5],
    'xgbclassifier__learning_rate': [0.01, 0.1]
}

model_pipeline = make_pipeline(preprocessor, xgb_model)

with mlflow.start_run():
    grid_search = GridSearchCV(model_pipeline, param_grid, cv=3, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    # Log all the tuned parameters (Nested runs)
    results = grid_search.cv_results_
    for i in range(len(results['params'])):
        param_set = results['params'][i]
        mean_score = results['mean_test_score'][i]

        with mlflow.start_run(nested=True):
            mlflow.log_params(param_set)
            mlflow.log_metric("mean_accuracy", mean_score)

    # Log Best model
    mlflow.log_params(grid_search.best_params_)
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metrics({
        "test_accuracy": accuracy,
        "test_f1_score": f1
    })
    print(f"Best Params: {grid_search.best_params_}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test F1: {f1:.4f}")

Loading train and test splits from Hugging Face...
Data loaded successfully from Hugging Face Hub.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:39:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:39:09] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:39:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:39:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:

Best Params: {'xgbclassifier__learning_rate': 0.1, 'xgbclassifier__max_depth': 5, 'xgbclassifier__n_estimators': 100}
Test Accuracy: 0.9056
Test F1: 0.7153


### Experimentation and Tracking (Production Environment)
Script for CI/CD pipeline to train model and push to Hugging Face Model Hub.

In [36]:
%%writefile mlops_project/model_building/train.py
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.metrics import accuracy_score
from huggingface_hub import HfApi
import os
import joblib

repo_id = "ashwindatasense/wellness-tourism-dataset"
model_repo_id = "ashwindatasense/wellness-tourism-model"

api = HfApi(token=os.getenv("HF_TOKEN"))

# Create Model Repo if not exists
try:
    api.repo_info(repo_id=model_repo_id, repo_type="model")
except Exception:
    api.create_repo(repo_id=model_repo_id, repo_type="model", private=False)

# Construct direct download URLs from Hugging Face
base_url = f"https://huggingface.co/datasets/{repo_id}/resolve/main"

# Load Prepared Data using HTTPS instead of hf://
print("Loading prepared data from Hugging Face Hub...")
X_train = pd.read_csv(f"{base_url}/X_train.csv")
X_test = pd.read_csv(f"{base_url}/X_test.csv")
y_train = pd.read_csv(f"{base_url}/y_train.csv").squeeze()
y_test = pd.read_csv(f"{base_url}/y_test.csv").squeeze()

numeric_features = ['Age', 'DurationOfPitch', 'NumberOfPersonVisiting', 'NumberOfFollowups',
                    'PreferredPropertyStar', 'NumberOfTrips', 'PitchSatisfactionScore',
                    'NumberOfChildrenVisiting', 'MonthlyIncome']
categorical_features = ['TypeofContact', 'Occupation', 'Gender', 'ProductPitched', 'MaritalStatus', 'Designation']
passthrough_features = ['CityTier', 'Passport', 'OwnCar']

numeric_transformer = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))
passthrough_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'))

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features),
    (passthrough_transformer, passthrough_features)
)

xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=100, max_depth=5, learning_rate=0.1)
model_pipeline = make_pipeline(preprocessor, xgb_model)

print("Training production model...")
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)
print(f"Prod Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Save and upload model
joblib.dump(model_pipeline, "model.joblib")
api.upload_file(
    path_or_fileobj="model.joblib",
    path_in_repo="model.joblib",
    repo_id=model_repo_id,
    repo_type="model"
)
print("Model uploaded successfully!")

Overwriting mlops_project/model_building/train.py


## Model Deployment
This section contains the necessary scripts for deploying the model via a Docker container on Hugging Face Spaces using Streamlit.

In [37]:
import os
os.makedirs("mlops_project/deployment", exist_ok=True)

In [38]:
%%writefile mlops_project/deployment/requirements.txt
streamlit==1.32.0
pandas
scikit-learn
xgboost
huggingface-hub
joblib


Overwriting mlops_project/deployment/requirements.txt


In [39]:
%%writefile mlops_project/deployment/Dockerfile
FROM python:3.9-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8501

CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]


Overwriting mlops_project/deployment/Dockerfile


In [40]:
%%writefile mlops_project/deployment/app.py
import streamlit as st
import pandas as pd
import joblib
from huggingface_hub import hf_hub_download
import os

st.set_page_config(page_title="Wellness Tourism Predictor", layout="centered")
st.title("Wellness Tourism Package Purchase Predictor")
st.markdown("Enter the customer details below to predict if they will purchase the new Wellness Tourism Package.")

@st.cache_resource
def load_model():
    model_path = hf_hub_download(repo_id="ashwindatasense/wellness-tourism-model", filename="model.joblib")
    return joblib.load(model_path)

try:
    model = load_model()
except Exception as e:
    st.error(f"Failed to load model from Hugging Face: {e}")
    st.stop()

# Layout
col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Age", min_value=18, max_value=100, value=30)
    duration_of_pitch = st.number_input("Duration of Pitch (minutes)", min_value=1, max_value=120, value=15)
    num_person = st.number_input("Number of Persons Visiting", min_value=1, max_value=10, value=2)
    num_followups = st.number_input("Number of Followups", min_value=1, max_value=10, value=3)
    pref_star = st.selectbox("Preferred Property Star", [3.0, 4.0, 5.0])
    num_trips = st.number_input("Number of Trips", min_value=1, max_value=20, value=2)
    pitch_sat = st.selectbox("Pitch Satisfaction Score", [1, 2, 3, 4, 5])
    num_children = st.number_input("Number of Children Visiting", min_value=0, max_value=10, value=0)
    monthly_income = st.number_input("Monthly Income", min_value=1000.0, max_value=200000.0, value=20000.0)

with col2:
    contact = st.selectbox("Type of Contact", ["Self Enquiry", "Company Invited"])
    occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Large Business", "Free Lancer"])
    gender = st.selectbox("Gender", ["Male", "Female"])
    product = st.selectbox("Product Pitched", ["Basic", "Standard", "Deluxe", "Super Deluxe", "King"])
    marital = st.selectbox("Marital Status", ["Single", "Married", "Divorced", "Unmarried"])
    designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])
    city_tier = st.selectbox("City Tier", [1, 2, 3])
    passport = st.selectbox("Has Passport?", ["Yes", "No"])
    own_car = st.selectbox("Owns Car?", ["Yes", "No"])

predict_btn = st.button("Predict Purchase Probability")

if predict_btn:
    # Convert inputs to dataframe
    input_data = pd.DataFrame({
        'Age': [age],
        'DurationOfPitch': [duration_of_pitch],
        'NumberOfPersonVisiting': [num_person],
        'NumberOfFollowups': [num_followups],
        'PreferredPropertyStar': [pref_star],
        'NumberOfTrips': [num_trips],
        'PitchSatisfactionScore': [pitch_sat],
        'NumberOfChildrenVisiting': [num_children],
        'MonthlyIncome': [monthly_income],
        'TypeofContact': [contact],
        'Occupation': [occupation],
        'Gender': [gender],
        'ProductPitched': [product],
        'MaritalStatus': [marital],
        'Designation': [designation],
        'CityTier': [city_tier],
        'Passport': [1 if passport == "Yes" else 0],
        'OwnCar': [1 if own_car == "Yes" else 0]
    })

    prediction = model.predict(input_data)[0]
    proba = model.predict_proba(input_data)[0][1]

    st.markdown("---")
    if prediction == 1:
        st.success(f"**High Likelihood of Purchase!** (Probability: {proba:.2%})")
        st.balloons()
    else:
        st.warning(f"**Low Likelihood of Purchase.** (Probability: {proba:.2%})")


Overwriting mlops_project/deployment/app.py


In [41]:
%%writefile mlops_project/deployment/deploy.py
from huggingface_hub import HfApi
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

api = HfApi(token=hf_token)
user_info = api.whoami()
username = user_info['name']

space_id = f"{username}/wellness-tourism-app"

print(f"Checking if Space '{space_id}' exists...")
try:
    api.repo_info(repo_id=space_id, repo_type="space")
    print("Space already exists.")
except Exception:
    print("Creating new Streamlit Space...")
    api.create_repo(
        repo_id=space_id,
        repo_type="space",
        space_sdk="docker",
        private=False,
        token=hf_token
    )
    print("Space created!")

print("Uploading deployment files...")
api.upload_folder(
    folder_path="mlops_project/deployment",
    repo_id=space_id,
    repo_type="space",
    token=hf_token
)
print("Deployment files uploaded successfully! Your app is building on Hugging Face Spaces.")

Overwriting mlops_project/deployment/deploy.py


## MLOps Pipeline with Github Actions Workflow
Automates the entire MLOps workflow (Data Registration, Preparation, Model Training, and Deployment) via GitHub Actions when code is pushed to the repository.

In [42]:
import os
os.makedirs(".github/workflows", exist_ok=True)

In [43]:
%%writefile .github/workflows/pipeline.yml
name: Wellness Tourism MLOps Pipeline

on:
  push:
    branches:
      - main

jobs:
  mlops-pipeline:
    runs-on: ubuntu-latest
    permissions:
      contents: write

    steps:
    - name: Checkout Code
      uses: actions/checkout@v3
      with:
        token: ${{ secrets.GITHUB_TOKEN }}

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.9'

    - name: Install Dependencies
      run: |
        pip install -r mlops_project/deployment/requirements.txt
        pip install python-dotenv

    - name: Execute Data Registration
      env:
        HF_TOKEN: ${{ secrets.HF_TOKEN }}
      run: python mlops_project/model_building/data_register.py

    - name: Execute Data Preparation
      env:
        HF_TOKEN: ${{ secrets.HF_TOKEN }}
      run: python mlops_project/model_building/prep.py

    - name: Execute Model Training
      env:
        HF_TOKEN: ${{ secrets.HF_TOKEN }}
      run: python mlops_project/model_building/train.py

    - name: Execute Model Deployment
      env:
        HF_TOKEN: ${{ secrets.HF_TOKEN }}
      run: python mlops_project/deployment/deploy.py

    - name: Push code updates to main branch
      run: |
        git config --global user.name 'github-actions[bot]'
        git config --global user.email 'github-actions[bot]@users.noreply.github.com'
        git add .
        git diff --quiet && git diff --staged --quiet || (git commit -m "Automated model pipeline update" && git push)


Overwriting .github/workflows/pipeline.yml
